<a href="https://colab.research.google.com/github/buketdede/healthcare-appointment-no-show-analysis-/blob/main/healthcare_noshows_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/healthcare_noshows_appt.csv")

print("Rows and columns:", df.shape)
print(df.head())
print(df.shape)
print(df.info())

Rows and columns: (106987, 15)
      PatientId  AppointmentID Gender ScheduledDay AppointmentDay  Age  \
0  2.987250e+13        5642903      F   2016-04-29     2016-04-29   62   
1  5.589978e+14        5642503      M   2016-04-29     2016-04-29   56   
2  4.262962e+12        5642549      F   2016-04-29     2016-04-29   62   
3  8.679512e+11        5642828      F   2016-04-29     2016-04-29    8   
4  8.841186e+12        5642494      F   2016-04-29     2016-04-29   56   

       Neighbourhood  Scholarship  Hipertension  Diabetes  Alcoholism  \
0    JARDIM DA PENHA        False          True     False       False   
1    JARDIM DA PENHA        False         False     False       False   
2      MATA DA PRAIA        False         False     False       False   
3  PONTAL DE CAMBURI        False         False     False       False   
4    JARDIM DA PENHA        False          True      True       False   

   Handcap  SMS_received  Showed_up  Date.diff  
0    False         False       True 

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nAge range:")
print(df["Age"].min(), df["Age"].max())

print("\nWaiting-time range:")
print(df["Date.diff"].min(), df["Date.diff"].max())

Missing values:
PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
Showed_up         0
Date.diff         0
dtype: int64

Duplicate rows:
0

Age range:
1 115

Waiting-time range:
-6 179


The dataset contains 106,987 appointments with no missing values or exact duplicate rows. Five appointments have negative waiting periods and require cleaning.

In [ ]:
df = df.rename(columns={
    "Hipertension": "Hypertension",
    "Handcap": "Handicap",
    "Date.diff": "WaitingDays"
})

In [ ]:
df["ScheduledDay"] = pd.to_datetime(df["ScheduledDay"])
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"])

print(df[["ScheduledDay", "AppointmentDay"]].dtypes)

ScheduledDay      datetime64[ns]
AppointmentDay    datetime64[ns]
dtype: object


In [ ]:
df["ScheduledDay"] = pd.to_datetime(df["ScheduledDay"])
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"])
print(df[["ScheduledDay", "AppointmentDay"]].dtypes)

ScheduledDay      datetime64[ns]
AppointmentDay    datetime64[ns]
dtype: object


In [ ]:
df = df[df["WaitingDays"] >= 0].copy()

In [ ]:
df[df["Age"] > 100]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,Showed_up,WaitingDays
56030,9.762948e+14,5651757,F,2016-05-03,2016-05-03,102,CONQUISTA,False,False,False,False,False,False,True,0
61803,3.196321e+13,5700278,F,2016-05-16,2016-05-19,115,ANDORINHAS,False,False,False,False,True,False,False,3
61806,3.196321e+13,5700279,F,2016-05-16,2016-05-19,115,ANDORINHAS,False,False,False,False,True,False,False,3
65876,3.196321e+13,5562812,F,2016-04-08,2016-05-16,115,ANDORINHAS,False,False,False,False,True,False,False,38
73825,3.196321e+13,5744037,F,2016-05-30,2016-05-30,115,ANDORINHAS,False,False,False,False,True,False,True,0
87422,2.342836e+11,5751563,F,2016-05-31,2016-06-02,102,MARIA ORTIZ,False,False,False,False,False,False,True,2
94560,7.482346e+14,5717451,F,2016-05-19,2016-06-03,115,SÃO JOSÉ,False,True,False,False,False,True,True,15


In [ ]:
df["NoShow"] = (~df["Showed_up"]).astype(int)

In [ ]:
bins = [-1, 0, 3, 7, 14, 30, float("inf")]
labels = [
    "Same day",
    "1-3 days",
    "4-7 days",
    "8-14 days",
    "15-30 days",
    "31+ days"
]

df["WaitingGroup"] = pd.cut(
    df["WaitingDays"],
    bins=bins,
    labels=labels
)

In [ ]:
age_bins = [0, 17, 34, 49, 64, 150]
age_labels = ["1-17", "18-34", "35-49", "50-64", "65+"]

df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

In [ ]:
df["AppointmentWeekday"] = df["AppointmentDay"].dt.day_name()
df["AppointmentMonth"] = df["AppointmentDay"].dt.month_name()

In [ ]:
overall_rate = df["NoShow"].mean() * 100
print(f"Overall no-show rate: {overall_rate:.2f}%")

Overall no-show rate: 20.26%


In [ ]:
waiting_analysis = (
    df.groupby("WaitingGroup", observed=False)
      .agg(
          Appointments=("AppointmentID", "count"),
          NoShowRate=("NoShow", "mean")
      )
      .reset_index()
)

waiting_analysis["NoShowRate"] *= 100
print(waiting_analysis)

  WaitingGroup  Appointments  NoShowRate
0     Same day         37154    4.685902
1     1-3 days         14303   22.946235
2     4-7 days         17143   25.088958
3    8-14 days         11644   30.522157
4   15-30 days         16742   32.743997
5     31+ days          9996   33.163265


In [ ]:
from google.colab import files
files.download('healthcare_noshows_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
age_analysis = (
    df.groupby("AgeGroup", observed=False)
      .agg(
          Appointments=("AppointmentID", "count"),
          NoShowRate=("NoShow", "mean")
      )
      .reset_index()
)

age_analysis["NoShowRate"] *= 100
print(age_analysis)

  AgeGroup  Appointments  NoShowRate
0     1-17         23839   22.471580
1    18-34         24244   23.972942
2    35-49         21864   20.531467
3    50-64         22634   16.722630
4      65+         14401   15.498924


In [ ]:
sms_analysis = (
    df.groupby(
        ["WaitingGroup", "SMS_received"],
        observed=False
    )
    .agg(
        Appointments=("AppointmentID", "count"),
        NoShowRate=("NoShow", "mean")
    )
    .reset_index()
)

sms_analysis["NoShowRate"] *= 100
print(sms_analysis)

   WaitingGroup  SMS_received  Appointments  NoShowRate
0      Same day         False         37154    4.685902
1      Same day          True             0         NaN
2      1-3 days         False         13418   23.043673
3      1-3 days          True           885   21.468927
4      4-7 days         False          6655   26.867017
5      4-7 days          True         10488   23.960717
6     8-14 days         False          4811   33.901476
7     8-14 days          True          6833   28.142836
8    15-30 days         False          6536   37.025704
9    15-30 days          True         10206   30.001960
10     31+ days         False          3823   37.535967
11     31+ days          True          6173   30.455208


In [ ]:
df.to_csv(
    "healthcare_noshows_cleaned.csv",
    index=False
)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 106982 entries, 0 to 106986
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   PatientId           106982 non-null  float64       
 1   AppointmentID       106982 non-null  int64         
 2   Gender              106982 non-null  object        
 3   ScheduledDay        106982 non-null  datetime64[ns]
 4   AppointmentDay      106982 non-null  datetime64[ns]
 5   Age                 106982 non-null  int64         
 6   Neighbourhood       106982 non-null  object        
 7   Scholarship         106982 non-null  bool          
 8   Hypertension        106982 non-null  bool          
 9   Diabetes            106982 non-null  bool          
 10  Alcoholism          106982 non-null  bool          
 11  Handicap            106982 non-null  bool          
 12  SMS_received        106982 non-null  bool          
 13  Showed_up           106982 non-nul